# RealWaste Progres 4 — Mitigation Training & Validation Evaluation Runner
Menarik kode modular `src/` beserta hasil Fase 1 (`EXP-01..EXP-04`) dari GitHub, membangun cache `uint8` 128x128, menjalankan **eksperimen mitigasi overfitting & underfitting (`EXP-05..EXP-07`)** pada GPU Kaggle, serta menghasilkan evaluasi visual lengkap seluruh **7 eksperimen** pada **Validation Set**.

In [ ]:
import os, sys, platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
REPO_URL = 'https://github.com/tryaannn/klasifikasi-jenis-sampah.git'
BRANCH = 'test/majid'
REPO_DIR = '/kaggle/working/klasifikasi-jenis-sampah'

!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -n 2 --oneline


In [ ]:
# Verifikasi kontrak model & anti-leakage sebelum training
!python -m pytest -q

from pathlib import Path
from src.dataset import resolve_realwaste_root
rw_root = resolve_realwaste_root('/kaggle/input')
print('Resolved RealWaste root:', rw_root)


In [ ]:
import shutil
CACHE_PATH = '/kaggle/working/realwaste_128_uint8.npz'
OUT_RESULTS = '/kaggle/working/progress4_results'
OUT_FIGURES = '/kaggle/working/progress4_figures'

# Salin hasil Fase 1 (EXP-01..EXP-04) dari repo ke OUT_RESULTS agar evaluasi membandingkan EXP-01..EXP-07
shutil.copytree('results/progress4', OUT_RESULTS, dirs_exist_ok=True)
!python -m src.dataset --data-dir "{rw_root}" --cache-out "{CACHE_PATH}"


In [ ]:
# Eksekusi 3 eksperimen mitigasi (EXP-05, EXP-06, EXP-07)
!python -m src.train --experiments mitigation \
    --cache-path "{CACHE_PATH}" \
    --output-dir "{OUT_RESULTS}" \
    --patience 10 --seed 42


In [ ]:
# Evaluasi checkpoint terbaik dari semua 7 eksperimen KHUSUS pada Validation Set (713 citra)
!python -m src.evaluate_val \
    --results-dir "{OUT_RESULTS}" \
    --figures-dir "{OUT_FIGURES}" \
    --cache-path "{CACHE_PATH}"


In [ ]:
import json, os
from pathlib import Path
%cd /kaggle/working
if os.path.exists(CACHE_PATH):
    os.remove(CACHE_PATH)
!rm -rf /kaggle/working/klasifikasi-jenis-sampah

comp = json.loads(Path(f'{OUT_RESULTS}/experiment_comparison.json').read_text())
print(json.dumps(comp, indent=2))
